# SIM V1 3D — Phase B: dataset generation

3-D analog of `SIM/phase_b_dataset.py`, as a Colab notebook. For N transmitter
positions sampled over `valid_tx` (min 3-D spacing) × the manifest bands, it runs
`SceneV3` and stores the **normalized PL volume** as the training target for
Phase C.

- **target**: PL clipped to `[pl_min, pl_min+pl_range]` → `/pl_range` → `[0,1]`, float16
- **inputs**: `tx` voxel + `freq_feat` (the geometry is FIXED, stored once — not per sample)
- **splits**: by position, stratified by X-Z octant, seed 0, written once
- **shards**: resumable `.npz` — re-run the generate cell to resume after a disconnect

Runs on a **CPU** runtime (`SceneV3` is NumPy, ~1 s/sample). Upload the
`SIM V1 3D/` folder (with the `*.npy` grids, `manifest_3d.json`, `engine_3d.py`)
to Drive and set `ROOT`.


In [1]:
import sys, os, json, glob, numpy as np
ROOT = "/content/drive/MyDrive/SIM V1 3D"   # <-- edit (or "." if running inside the folder)
try:
    from google.colab import drive; drive.mount("/content/drive")
except Exception:
    pass
sys.path.insert(0, ROOT)
from engine_3d import load_scene            # SceneV3 lives next to this notebook

scene, man = load_scene(ROOT)
valid = np.load(f"{ROOT}/valid_tx_mask.npy")
freqs = man["freqs_mhz"]; norm = man["norm"]
print("grid", scene.M.shape, "| bands", freqs, "| valid Tx voxels", int(valid.sum()))

Mounted at /content/drive


ModuleNotFoundError: No module named 'engine_3d'

## Config

`SMOKE = True` does a quick 8-position run to check the whole path end-to-end;
set it to `False` for the full dataset. Size note: grid ≈ 340k voxels → fp16
target ≈ 0.68 MB/sample, so 1500 × 4 ≈ 6000 samples ≈ a few GB — lower `N_POSITIONS`
or trim `freqs` to shrink it.

In [ ]:
SMOKE        = True
N_POSITIONS  = 1500
MIN_SPACING  = 3.0     # voxels
SHARD_POS    = 50      # × len(freqs) samples per shard
SEED         = 0

pl_lo, pl_rng = norm["pl_min_db"], norm["pl_range_db"]
f_lo, f_hi = np.log10(norm["freq_log_lo_mhz"]), np.log10(norm["freq_log_hi_mhz"])
OUT = f"{ROOT}/dataset"; os.makedirs(OUT, exist_ok=True)


def sample_positions(valid, rng, n, spacing):
    cells = np.argwhere(valid); rng.shuffle(cells)
    taken = np.zeros(valid.shape, bool); r = int(np.ceil(spacing)); out = []
    for x, y, z in cells:
        sl = (slice(max(0, x-r), x+r+1), slice(max(0, y-r), y+r+1), slice(max(0, z-r), z+r+1))
        if taken[sl].any(): continue
        taken[x, y, z] = True; out.append((int(x), int(y), int(z)))
        if len(out) == n: break
    return np.array(out)


def octant_of(pos, shape):
    nx, ny, nz = shape
    ox = np.clip(pos[:, 0]*4//nx, 0, 3); oz = np.clip(pos[:, 2]*2//nz, 0, 1)
    return (oz*4 + ox).astype(int)


def make_splits(pos, shape, rng):
    octs = octant_of(pos, shape); idx = {"train": [], "val": [], "test": []}
    for o in range(8):
        ids = np.nonzero(octs == o)[0]; rng.shuffle(ids); nv = nt = round(len(ids)*0.1)
        idx["val"] += ids[:nv].tolist(); idx["test"] += ids[nv:nv+nt].tolist(); idx["train"] += ids[nv+nt:].tolist()
    return {k: sorted(v) for k, v in idx.items()}

## Sample transmitter positions + build splits

Splits are keyed to `pos_id` and written once to `dataset/splits.json` (re-runs
reuse them, so train/val/test never leak across a resumed run).

In [ ]:
rng = np.random.default_rng(SEED)
n_pos = 8 if SMOKE else N_POSITIONS
pos = sample_positions(valid, rng, n_pos, MIN_SPACING); n_pos = len(pos)
print(f"{n_pos} Tx positions × {len(freqs)} bands = {n_pos*len(freqs)} samples")

split_file = f"{OUT}/splits.json"
if os.path.exists(split_file) and not SMOKE:
    splits = json.load(open(split_file))
else:
    splits = make_splits(pos, scene.M.shape, np.random.default_rng(SEED+1))
    if not SMOKE:
        json.dump(dict(seed=SEED, n_positions=n_pos, positions=pos.tolist(), **splits), open(split_file, "w"))
print({k: len(v) for k, v in splits.items() if k in ("train", "val", "test")})

## Generate shards (resumable)

Each shard holds `SHARD_POS` positions × all bands. Existing shards are skipped —
if Colab disconnects, just **re-run this cell** to pick up where it stopped.

In [ ]:
n_shards = int(np.ceil(n_pos / SHARD_POS))
for s in range(n_shards):
    sp = f"{OUT}/shard_{s:03d}.npz"
    if os.path.exists(sp):
        print(f"shard {s}: exists, skipping"); continue
    p0, p1 = s*SHARD_POS, min((s+1)*SHARD_POS, n_pos)
    tx_l, ff_l, tg_l, id_l = [], [], [], []
    for pi in range(p0, p1):
        x, y, z = pos[pi]
        for f in freqs:
            pl = scene.pathloss_map((float(x), float(y), float(z)), float(f))
            tgt = ((np.clip(pl, pl_lo, pl_lo+pl_rng) - pl_lo) / pl_rng).astype(np.float16)
            tx_l.append((x, y, z)); ff_l.append((np.log10(f)-f_lo)/(f_hi-f_lo))
            tg_l.append(tgt); id_l.append(pi)
    np.savez_compressed(sp, tx=np.array(tx_l, np.int16), freq_feat=np.array(ff_l, np.float32),
                        target=np.stack(tg_l), pos_id=np.array(id_l, np.int32))
    print(f"shard {s+1}/{n_shards} written ({p1-p0} positions)")

json.dump(dict(n_positions=n_pos, freqs_mhz=freqs, seed=SEED, min_spacing=MIN_SPACING,
               grid_shape=man["grid_shape"], cell_size_m=man["cell_size_m"], norm=norm),
          open(f"{OUT}/dataset_meta.json", "w"), indent=2)
print("dataset complete →", OUT)

## Sanity check — one generated sample

Load a shard and show a horizontal slice at the Tx height (denormalized to dB).
Loss should fall near the Tx and rise with distance / through walls.

In [ ]:
import matplotlib.pyplot as plt
d = np.load(sorted(glob.glob(f"{OUT}/shard_*.npz"))[0])
i = 0
vol = d["target"][i].astype(np.float32) * pl_rng + pl_lo     # back to dB
tx = d["tx"][i]; iy = int(tx[1])
inside = scene.inside
plm = np.where(inside, vol, np.nan)[:, iy, :].T
plt.figure(figsize=(9, 4))
plt.imshow(plm, origin="lower", cmap="viridis")
plt.plot(tx[0], tx[2], "r*", ms=14, mec="white")
plt.title(f"sample {i}: PL dB at iy={iy} (Tx {tx.tolist()})"); plt.colorbar(label="PL (dB)")
plt.tight_layout(); plt.show()
print("target range (dB):", float(vol[inside].min()), "…", float(vol[inside].max()))